# Project Overview & Objectives
Welcome to this walkthrough on Instruction Fine-Tuning. As a data scientist, you will often find that pre-trained Large Language Models (LLMs) are excellent at predicting the next word, but struggle to act as helpful assistants. They need to be taught how to respond to prompts.

In this notebook, we will transition a base, non-instructional model (TinyLlama) into a domain-specific assistant capable of answering Pharmaceutical questions. We will achieve this using Low-Rank Adaptation (LoRA), a Parameter-Efficient Fine-Tuning (PEFT) technique that allows us to train massive models on limited hardware.

**Key Objectives:**

- Understand the behavioral difference between a base model and an instruction-tuned model.

- Process and format raw domain data into a structured prompt format (Alpaca style).

- Implement Response Masking during tokenization to calculate loss exclusively on the generated response.

- Configure and train a LoRA adapter for efficient domain adaptation.

# Environment Setup

In [ ]:
# %pip install -U torch transformers datasets peft accelerate pandas

In [ ]:
%pip install -U "torchao>=0.16.0"

In [ ]:
from IPython.display import display

import pandas as pd
from datasets import load_dataset

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

from peft import LoraConfig, get_peft_model, TaskType, PeftModel

In [ ]:
# Mount Google Drive and set up paths

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ASSETS = Path('/content/drive/MyDrive/LLM-Fine-Tuning/DomainSpecific/assets')
DRIVE_ASSETS.mkdir(parents=True, exist_ok=True)
print("Drive assets folder:", DRIVE_ASSETS)

# Data Acquisition & Exploratory Data Analysis (EDA)

Context: In data science, we often work with various file formats and structures. Before touching our primary pharmaceutical dataset, let's briefly demonstrate how to load, format, and save a standard dataset (using a mental health conversational dataset as an example). This illustrates how we handle input/output operations and prepare raw data for LLM consumption.

## Step 1: Exploratory Formatting on a Sample Dataset

In [ ]:
# Load a sample conversational dataset
sample_dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

# Define a formatting function to structure the conversation
def format_conversational_row(example):
    question = example["Context"]
    answer = example["Response"]
    # Formatting using standard tags to separate context and response
    example["Text"] = f"[Context] {question} [/Response] {answer}"
    return example

In [ ]:
# Apply the formatting across the entire dataset
formatted_sample_dataset = sample_dataset.map(format_conversational_row)
display(formatted_sample_dataset[0]["Text"])

In [ ]:
# Convert to a Pandas DataFrame for easy I/O operations
df_sample = pd.DataFrame(sample_dataset)

# Save to multiple formats for pipeline versatility
df_sample.to_csv("mental_health_counseling_conversations.csv", index=False)
df_sample.to_json("mental_health_counseling_conversations.jsonl", orient="records", lines=True)

## Step 2: Loading the Target Domain Dataset

🚨 MAKE SURE TO UPLOAD THE DATASET TO DRIVE!!!

In [ ]:
# Load the pharmaceutical instruction data
# First upload the "pharma_instruction_data.csv" file to the Colab environment
pharma_dataset = load_dataset("csv", data_files=f"{DRIVE_ASSETS}/pharma_instruction_data.csv", split="train")
display(pharma_dataset)

# Preprocessing & Feature Engineering

Context: LLMs expect text to be presented in a highly consistent manner. We will format our data into the standard Alpaca Prompt Format. More importantly, we will apply **Response Masking**.

**Why do we mask?** 

During training, we only want to penalize the model for getting the answer wrong, not the question. By setting the labels of the instruction and input tokens to -100, PyTorch's Cross-Entropy loss function will ignore them, focusing the model's learning purely on generating the correct response.

In [ ]:
# Initialize Tokenizer for TinyLlama
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Ensure pad token is set (crucial for batching)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_alpaca_prompt(example):
    """
    Formats the raw row into a unified instruction-style prompt.
    """
    # Safeguard against CSV empty values
    in_text = example['input'] if example['input'] else ""
    
    # Explicitly add the EOS token at the end of the completion
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{in_text}\n### Response:\n{example['output']}{tokenizer.eos_token}"
    return {"text": prompt}

# Apply formatting
pharma_dataset = pharma_dataset.map(format_alpaca_prompt)
display(pharma_dataset[0]["text"])


In [ ]:
def tokenize_and_mask(example):
    """
    Tokenizes the text and masks the instruction/input portions so the loss is only calculated on the generated response.
    """
    text = example["text"]

    # Tokenize full text with padding and truncation
    encodings = tokenizer(text, truncation=True, padding="max_length", max_length=512)
    input_ids = encodings["input_ids"]

    # Locate the exact position where the response begins
    response_marker = "### Response:"
    response_start_idx = text.find(response_marker)

    if response_start_idx != -1:
        # Calculate how many tokens make up the instruction/input
        response_token_start = len(tokenizer(text[:response_start_idx])["input_ids"])
    else:
        response_token_start = 0 

    # Clone the input_ids to create our labels
    labels = input_ids.copy()
    
    # 1. Mask out everything before the response with -100
    labels[:response_token_start] = [-100] * response_token_start

    # 2. Mask the padding tokens so it doesn't learn to only predict EOS
    attention_mask = encodings["attention_mask"]
    for i in range(len(attention_mask)):
        if attention_mask[i] == 0:
            labels[i] = -100

    encodings["labels"] = labels
    return encodings

# Apply tokenization and masking to the entire dataset
tokenized_pharma_dataset = pharma_dataset.map(tokenize_and_mask, batched=False)


# Model Development & Training

Context: Here we configure our LoRA adapter, specifying which linear layers to target (q_proj, v_proj), and initialize our base model. We then set our TrainingArguments to define our learning rate, batch size, and precision (fp16 for faster GPU training).

## Model Selection & Architecture Overview

**Base Model (TinyLlama 1.1B):**

Pros: Extremely lightweight, fast to train, fits easily into consumer GPUs (like the T4 provided in Colab).

Cons: Lower parameter count means it lacks the deep emergent reasoning of larger models (like Llama-3 70B), requiring strict, clean data to perform well.

**Fine-Tuning Strategy (LoRA):**
Instead of updating all 1.1 billion parameters, LoRA freezes the original weights $W_0$ and injects trainable rank decomposition matrices. The weight update is calculated as $W = W_0 + \Delta W = W_0 + BA$, where $B$ and $A$ are low-rank matrices. This reduces the number of trainable parameters by over 99%, preventing catastrophic forgetting and drastically reducing VRAM usage.

In [ ]:
# Define LoRA Configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,  # Rank of the LoRA matrices
    lora_alpha=16,  # Scaling factor
    lora_dropout=0.05,  # Regularization to prevent overfitting
    target_modules=["q_proj", "v_proj"],  # Attention layers to adapt
    bias="none"
)

# Load base model (non-instructional trained)
base_model_path = f"{DRIVE_ASSETS}/non-instruction-tinyllama-model"  # Replace with actual path
non_instructional_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    dtype=torch.float16,
    device_map="auto"
)

# Wrap the base model with PEFT/LoRA
peft_model = get_peft_model(non_instructional_model, lora_config)

# Define Training Hyperparameters
training_args = TrainingArguments(
    output_dir="./instruction-tinyllama",
    num_train_epochs=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    fp16=True,  # Half-precision training for speed/memory efficiency
    logging_steps=10,
    save_total_limit=1,
    report_to="none"
)

# Initialize the Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_pharma_dataset,
)

In [ ]:
# Execute the training loop
trainer.train()

# Save the fine-tuned adapter and tokenizer
trainer.save_model(f"{DRIVE_ASSETS}/instruction-tinyllama-adapter")
tokenizer.save_pretrained(f"{DRIVE_ASSETS}/instruction-tinyllama-tokenizer")

# Evaluation & Results

Context: A critical step in model development is qualitative evaluation. We will now iterate through a list of domain-specific questions, passing them through both the Base Model and our newly trained Instruction-Tuned Model to observe the behavioral shift.

In [ ]:
# Load a clean instance of the base model for accurate comparison
clean_base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    dtype=torch.float16,
    device_map="auto"
)

# Load the fine-tuned adapter on top of the clean base model
tuned_model = PeftModel.from_pretrained(clean_base_model, f"{DRIVE_ASSETS}/instruction-tinyllama-adapter")

In [ ]:
# Define test queries
evaluation_questions = [
    "Explain the mechanism of action of Metformin.",
    # "List two advantages of combining Atorvastatin with Ezetimibe.",
    # "Summarize how mRNA vaccines work and mention one current research focus."
]

# Run Inference Comparison
for query in evaluation_questions:
    print(f"Question: {query}")
    
    # -----------------------------------------------------
    # 1. Non-Instruction Model Evaluation
    # -----------------------------------------------------
    print("\n--- Non-Instruction Model ---")
    inputs = tokenizer(query, return_tensors="pt").to("cuda")
    outputs_base = clean_base_model.generate(
        **inputs, 
        max_new_tokens=80,
        pad_token_id=tokenizer.eos_token_id
    )
    print(tokenizer.decode(outputs_base[0], skip_special_tokens=True))

    # -----------------------------------------------------
    # 2. Instruction-Tuned Model Evaluation
    # -----------------------------------------------------
    print("\n--- Instruction-Tuned Model ---")
    # Wrap the query in the exact format the model was trained on
    prompt = f"### Instruction:\n{query}\n### Input:\n\n### Response:\n"
    inputs_tuned = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs_tuned = tuned_model.generate(
        **inputs_tuned, 
        max_new_tokens=100,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
    print(tokenizer.decode(outputs_tuned[0], skip_special_tokens=True))

# Conclusion & Future Work

In this project, we successfully established an instruction fine-tuning pipeline. We learned how to manipulate data into a strict schema, utilized response masking to isolate our loss calculations, and injected LoRA weights to adapt a generic causal language model into a specialized pharmaceutical assistant.

**Next Steps to Consider:**

- Hyperparameter Tuning: Experiment with increasing the LoRA rank (r=16 or r=32) to capture more complex domain nuances.

- Quantitative Evaluation: Implement a framework like ROUGE, BLEU, or an LLM-as-a-judge system to mathematically score the model's outputs against a holdout test set.

- Preference Alignment: To further refine the assistant's tone and safety, the next architectural step would be to apply Reinforcement Learning from Human Feedback (RLHF) or Direct Preference Optimization (DPO).